# Session 2 · Commands Primer

The runnable companion to the `S2_commands_primer` slide deck. Every command on the deck runs here against the real TESDA files, after a short Python and pandas refresher for anyone who already programs in another language such as Java, C#, SQL, or R.

The goal is fluency with the *tools*: reading a messy file into a table, joining tables, cleaning columns, and putting points on a map. The geospatial analysis itself is built on top of these in the Day 1 and Day 2 notebooks. Data streams from the public bucket `https://storage.googleapis.com/tesda-datasets/data/`.

**How to use this notebook**

- ▶ run the cells in order; each one builds on the tables loaded above it.
- Read the comments in the code cells. A comment appears only where a line does something worth explaining, often with the equivalent idea in another language.
- ✏️ fill the code between `### START CODE HERE ###` and `### END CODE HERE ###`.
- ✅ run the check cell after an exercise to confirm the answer.
- ❓ answer the short question with a computed value.

In [ ]:
# Run once per session (skip if already installed).
!pip -q install pandas geopandas folium matplotlib openpyxl pyarrow

## A short Python and pandas refresher

Python reads like pseudocode. A name is just a label on a value, with no type declaration, and indentation marks a block where other languages use braces. The next cell shows the handful of building blocks the rest of the notebook relies on.

In [ ]:
n_regions = 17                       # no type keyword; the name simply points at a value
regions = ["NCR", "Region III", "Region IV-A"]   # a list, written with [], indexed from 0
print(regions[0])                    # -> NCR   (first element, like array[0] elsewhere)

capacity = {"NCR": 919, "Region III": 845}        # a dict: keys mapped to values (a HashMap / JSON object)
print(capacity["NCR"])              # look a value up by its key

def per_capita(centers, people):     # def starts a function; the indented lines are its body
    return centers / people * 100_000   # 100_000 is just 100000; underscores aid reading

print(round(per_capita(919, 13_480_000), 2))

### The DataFrame is a table

A pandas `DataFrame` is a table held in memory, the same mental model as a SQL table, an R `data.frame`, or a spreadsheet. One column is a `Series`. A row is addressed by its index, a column by its name. A missing value is `NaN`, the counterpart of SQL `NULL`.

In [ ]:
import pandas as pd

demo = pd.DataFrame({            # build a small table from a dict of columns
    "region": ["NCR", "Region III", "Region IV-A"],
    "centers": [919, 845, 962],
})
print(demo.shape)                # (rows, columns), here (3, 2)
print(demo["centers"].sum())     # a column is a Series; it has vectorized methods like .sum()
demo[demo["centers"] > 900]      # a boolean mask filters rows, the way SQL WHERE does

**Expected output:** `(3, 2)`, then `2726`, then the two regions with more than 900 centers (NCR and Region IV-A).

## One base URL

Every file sits in the same bucket. Setting the base once keeps each read short. The helper percent-encodes the spaces in the long file names.

In [ ]:
import pandas as pd
import geopandas as gpd
from urllib.parse import quote

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
def src(name):
    return BASE + quote(name)       # turn spaces into %20 for an HTTP read
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 55)

# Part 1 - Data

Pull the scattered files into tables, then join and reshape them. One function per file format, the same idea each time.

## Load

Each format has its own reader. `dtype=str` tells pandas to leave every value as text instead of guessing a type, which matters for codes with leading zeros. The parquet file already carries its types, so it needs no such hint.

In [ ]:
centers = pd.read_csv(src("2026 Accredited Compentency Assessment Center based on "
                          "July 15, 2026 database.csv"), dtype=str)   # everything as text, cast later
coords = pd.read_parquet(BASE + "tesda_coordinates.parquet")           # geocoded institutions, typed already

print("centers:", centers.shape, "| coords:", coords.shape)
centers.head(3)

**Expected output:** centers about `(12871, 16)` and coords about `(8007, 21)`, then the first three accreditation rows.

## Index

pandas numbers the rows `0, 1, 2 ...` automatically. Reading a file that was saved with its index leaves a junk `Unnamed: 0` column, which `index_col=0` reclaims. Setting a meaningful index, such as the institution ID, lets a row be found by that value instead of by position.

In [ ]:
print("default index:", coords.index[:3].tolist())        # 0, 1, 2 unless told otherwise
by_id = coords.set_index("tesda_inst_id")                  # look a center up by its ID, not row number
print("rows now keyed by id, e.g.:", by_id.index[:3].tolist())
# by_id.loc["<some id>"] would now return that institution's row directly

## Merge

`merge` is a SQL `JOIN` between two tables on a shared key. The accreditation file has no institution ID, so the join runs on a key built from a normalized name and address. `how="left"` keeps every center row even when no match is found, and `validate="many_to_one"` raises an error if the right table has duplicate keys that would secretly multiply rows.

In [ ]:
def normalize(text):
    # lower-case and strip everything but letters and digits, so spacing and punctuation stop
    # blocking a match ("Dr. Jose Rizal MNHS" -> "drjoserizalmnhs")
    return text.fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

centers["link_key"] = normalize(centers["Assessment Center"]) + "|" + normalize(centers["Address"])
coords["link_key"] = normalize(coords["name"]) + "|" + normalize(coords["address"])

key_to_id = coords.drop_duplicates("link_key")[["link_key", "tesda_inst_id"]]   # one row per key
linked = centers.merge(key_to_id, on="link_key", how="left", validate="many_to_one")

print("accreditations:", len(linked))
print("matched to an institution id:", linked["tesda_inst_id"].notna().sum())

**Expected output:** about 12,871 accreditation rows, with roughly 11,000 matched to an institution ID. The unmatched remainder is the real cost of having no shared key.

## Join by location

A spatial join matches rows by *where they are* rather than by a shared value, for example which barangay a point falls inside. It needs both sides as `GeoDataFrame`s in the same coordinate system. A small 60-polygon sample keeps the read quick; the full file is read in `S2_D1_02_admin_boundaries`.

In [ ]:
import os
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg"   # let GDAL read the file over HTTP
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
brgy = gpd.read_file("/vsicurl/" + BASE + "consolidated_geodata_matched.gpkg", rows=60)

# use each barangay's own interior point as a stand-in for a center, so every join is guaranteed
# to land somewhere; a real run would use the institution coordinates
sample = brgy[["name", "geometry"]].copy()
sample["geometry"] = brgy.geometry.representative_point()

placed = gpd.sjoin(sample, brgy[["psgc_code", "geometry"]], predicate="within", how="left")
placed[["name", "psgc_code"]].head()

**Expected output:** each sampled point carries the `psgc_code` of the barangay it sits in, the result a point-in-polygon join produces.

## Melt

`melt` reshapes a wide table into a long one, turning spread-out columns into rows so that one row is one observation. Groupings, joins, and most plots expect the long shape.

In [ ]:
import pandas as pd
centers["pin_ok"] = (pd.to_numeric(centers["Latitude"], errors="coerce").between(4, 21)
                      .map({True: "has_pin", False: "no_pin"}))

wide = centers.groupby(["Region Name", "pin_ok"]).size().unstack(fill_value=0)   # region x pin status
long = wide.reset_index().melt(id_vars="Region Name", var_name="pin_ok", value_name="n")
print("wide:", wide.shape, "-> long:", long.shape)
long.head()

**Expected output:** a wide table of about `(17, 2)` becoming a long table of about `(34, 3)`, two rows per region.

## Fill missing

`NaN` marks a missing value. Each gap needs a decision: drop the row, flag it, or substitute a fallback. `pd.to_numeric(..., errors="coerce")` turns a blank or garbage coordinate into `NaN` instead of raising, which makes the gaps countable.

In [ ]:
lat = pd.to_numeric(centers["Latitude"], errors="coerce")   # blanks and junk -> NaN, no crash
print("missing or unparseable latitudes:", lat.isna().sum())

lat_filled = lat.fillna(lat.median())   # crude fallback; the audit notebook records a flag instead
print("after fillna, missing:", lat_filled.isna().sum())

# Part 2 - Cleaning

Real columns arrive dirty. A few recurring fixes decide whether a later join or map works at all. The audit notebook `S2_D1_03_institutions_and_coordinate_audit` carries the full treatment.

## Data type

pandas infers one type per column from the first rows it sees. Mixed text and numbers fall back to `object`, an integer column with one blank becomes `float`, and dates stay plain strings. Reading as text and casting on purpose keeps control.

In [ ]:
print(centers["Date of Expiry"].dtype)                       # object: still a string
expires = pd.to_datetime(centers["Date of Expiry"], errors="coerce")   # string -> real datetime
print(expires.dtype, "| earliest:", expires.min(), "latest:", expires.max())

## Whitespace

A trailing space makes `"TVET "` and `"TVET"` unequal to a join even though they look the same. `str.strip` and a regex collapse the stray spacing.

In [ ]:
raw = pd.Series(["  Manila", "Manila ", "Manila"])
clean = raw.str.strip().str.replace(r"\s+", " ", regex=True)   # trim ends, squeeze doubles
print("distinct before:", raw.nunique(), "| after:", clean.nunique())

## Coordinate system

Degrees of latitude and longitude (`EPSG:4326`) are for storing a position. Measuring a distance or an area needs a projection in meters, such as `EPSG:32651` for the Philippines. Measuring in degrees returns numbers that mean nothing on the ground.

In [ ]:
region3 = brgy.to_crs(32651)                       # reproject the 60 sample polygons to meters
area_km2 = region3.geometry.area / 1e6             # square meters -> square kilometers
print("a sample barangay area (km2):", round(area_km2.iloc[0], 3))

## ✏️ Exercise - centers per region

Build `region_counts`, a `Series` of how many accreditation rows fall in each region, sorted from most to fewest. Use the `Region Name` column of `centers`.

**Hint:** `value_counts()` counts and sorts in one call.

In [ ]:
### START CODE HERE ###
region_counts = None
### END CODE HERE ###
print(region_counts)

## ✅ Check the result

In [ ]:
ref = centers["Region Name"].value_counts()
assert region_counts is not None, "assign region_counts in the exercise cell"
assert region_counts.equals(ref), "counts or order do not match value_counts()"
print("\u2705 Passed! Most accreditations:", region_counts.index[0], "with", f"{region_counts.iloc[0]:,}")

## ❓ Question

After the `merge` above, how many accreditation rows ended up with no institution ID? Assign the integer to `q_unmatched`.

In [ ]:
### START CODE HERE ###
q_unmatched = None
### END CODE HERE ###

expected = int(linked["tesda_inst_id"].isna().sum())
assert q_unmatched == expected, f"not yet - {expected} rows have no id"
print("\u2705 Correct:", f"{expected:,}", "accreditations without an institution id")

# Part 4 - Map

A map is the fastest check on a geocoded table. A point in the sea is a bad coordinate the columns alone would not reveal. Only the pins the audit marked `valid` are drawn.

In [ ]:
import folium
located = coords[coords["coord_status"] == "valid"]
m = folium.Map(location=[12.8, 122.0], zoom_start=5, tiles="cartodbpositron")
for lat, lon in located[["latitude", "longitude"]].head(500).values:   # first 500 to keep it light
    folium.CircleMarker([lat, lon], radius=2, color="#d1495b", fill=True).add_to(m)
m

## Recap

- A `DataFrame` is a table; a column is a `Series`; `NaN` is the missing value.
- `read_csv`, `read_excel`, `read_parquet`, and `read_file` load the four formats; `dtype=str` stops pandas guessing and losing leading zeros.
- `merge` joins on a shared key (a SQL `JOIN`); `sjoin` joins by location.
- `melt` reshapes wide to long; `fillna` resolves gaps; `to_datetime` and `to_numeric` fix types.
- A projected CRS in meters is required before any distance or area.

Next, `S2_D1_00_data_sources` loads these files properly into one schema, and `S2_D1_03_institutions_and_coordinate_audit` cleans the coordinates in full.

## References & further reading

- pandas getting started for people who know another language: https://pandas.pydata.org/docs/getting_started/comparison/index.html
- pandas `merge`, `join`, `concat`: https://pandas.pydata.org/docs/user_guide/merging.html
- GeoPandas spatial joins: https://geopandas.org/en/stable/gallery/spatial_joins.html
- Companion slides: `slides/S2_commands_primer.md`.